#  GPU Programming

The aim of the lab is understand GPU programming by implementing following tasks.
1. Write optimized matrix multiplication on GPU
2. 1x1 Convoultion is the most dominant operation in a modern deep neural network like Mobilenet_v1. Use optimized matrix multiplication for 1x1 Convolution
3. Benchmark throughput on a modern GPU hardware.



## Prerequisites

The following code installs triton. The DEVICE should show CUDA if GPU is selected as an accelerator. The Notebook needs CUDA support.

In [2]:
!pip install triton torchprofile -q
import torch
import triton
import triton.language as tl
import time

torch.manual_seed(0); torch.cuda.manual_seed_all(0)
DEVICE = triton.runtime.driver.active.get_active_torch_device()
print(DEVICE)

cuda:0


## Utilities

Following functions can be used for benchmarking your gpu implementation of matrix multiplication

In [3]:
# ---------------------------- Utils ----------------------------

def bench(fn, warmup=2, reps=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    end = time.time()
    return (end - start) * 1000 / reps  # ms

def gflops(N, time_ms):
    """GFLOPS for an N x N square matmul: FLOPs = 2 * N^3."""
    return 2 * N**3 / (time_ms * 1e-3) / 1e9

def gflops_gemm(M, N, K, time_ms):
    """GFLOPS for a general M x K @ K x N matmul: FLOPs = 2 * M * N * K."""
    return 2 * M * N * K / (time_ms * 1e-3) / 1e9

**Task 1** Find out the following information about the GPU from torch, datasheet, or wikipedia.
* GPU
* SRAM
* VRAM
* Streaming Multiprocessors
* Peak Performance (FP32)
* Peak Performance (FP16)

In [4]:
# TODO
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name

print(f"GPU                         : {GPU_NAME}")
print(f"Compute Capability          : {props.major}.{props.minor}")
print(f"Streaming Multiprocessors   : {props.multi_processor_count}")
print(f"VRAM (total)                : {props.total_memory / 1024**3:.2f} GiB")
print(f"Shared Memory / SM          : {props.shared_memory_per_block / 1024:.0f} KiB")
print(f"L2 Cache                    : {props.L2_cache_size / 1024**2:.2f} MiB")

# ---- Fill in from datasheet / wikipedia ----
PEAK_FP32_TFLOPS = 8.1    # e.g. T4 ~ 8.1, A100 ~ 19.5, H100 ~ 67
PEAK_FP16_TFLOPS = 65    # tensor-core

print(f"\nPeak FP32 (datasheet)       : {PEAK_FP32_TFLOPS} TFLOPS")
print(f"Peak FP16 (tensor cores)    : {PEAK_FP16_TFLOPS} TFLOPS")

GPU                         : Tesla T4
Compute Capability          : 7.5
Streaming Multiprocessors   : 40
VRAM (total)                : 14.56 GiB
Shared Memory / SM          : 48 KiB
L2 Cache                    : 4.00 MiB

Peak FP32 (datasheet)       : 8.1 TFLOPS
Peak FP16 (tensor cores)    : 65 TFLOPS


# Matrix Multiplication

A simple matrix multiplication $C[m,n] = \sum A[m,k] * B[k,n]$ loads each element of A and B once per output element. Following is a naive implementation in triton which multiplies two 64x64 square matrices.
Assumption:
* Square matrix
* fp32 datatype

**Task 2**
* Execution Model: Which part of the following code runs on CPU and GPU respectively?
* Execution Model: How many triton programs are launched?
* Execution Model: What is the program_id of triton program which calculates C[1,1]
* Memory Model: What is total number of load and store from one triton program
* Memory Model: What is the total number of floating point operation per triton program

In [5]:
@triton.jit
def simple_matmul_kernel(a_ptr, b_ptr, c_ptr, N):
    pid_m = tl.program_id(0)   # row index
    pid_n = tl.program_id(1)   # column index

    acc = 0.0

    for k in range(N):
        a_val = tl.load(a_ptr + pid_m * N + k)
        b_val = tl.load(b_ptr + k * N + pid_n)
        acc += a_val * b_val

    tl.store(c_ptr + pid_m * N + pid_n, acc)

def simple_matmul(a, b):
    N = a.shape[0]
    c = torch.empty((N, N), device=a.device, dtype=a.dtype)

    simple_matmul_kernel[(N, N)](
        a, b, c, N
    )

    return c

A = torch.rand((64, 64), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((64, 64), device=DEVICE, dtype=torch.float32) - 0.5
torch.testing.assert_close(simple_matmul(A, B), torch.matmul(A, B), atol=1e-2, rtol=0)

# Benchmarking Performance

**Task 3**
* Measure mean GPU kernel time for matrix multiplication (N=1024) in milliseconds.
* What is the throughput of the naive GPU implementation in gflops?
* Is it higher compared to Optimized CPU implementation in Lab 2?


In [6]:
N = 1024
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5


t = bench(lambda: simple_matmul(A, B))

# TODO:
print(f"Kernel execution time: {t} ms")
print(f"Gflops: {gflops(N, t)} ")

Kernel execution time: 452.11284160614014 ms
Gflops: 4.749884211142998 


# Optimization


## Tiled Matrix Multiplication

block level data reuse.

Assumptions:
* Square tiles


In [57]:
@triton.jit
def simple_tiled_matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    N,
    BLOCK: tl.constexpr,
):
    # ---- which tile of C this program computes ----
    pid = tl.program_id(0)

    tiles_per_row = N // BLOCK
    pid_m = pid // tiles_per_row
    pid_n = pid % tiles_per_row

    # ---- row/col indices of this tile ----
    offs_m = pid_m * BLOCK + tl.arange(0, BLOCK)
    offs_n = pid_n * BLOCK + tl.arange(0, BLOCK)

    # accumulator for C tile
    acc = tl.zeros((BLOCK, BLOCK), dtype=tl.float32)

    # ---- iterate over K tiles ----
    for k in range(0, N, BLOCK):

        offs_k = k + tl.arange(0, BLOCK)

        # load A and B tiles
        a = tl.load(a_ptr + offs_m[:, None] * N + offs_k[None, :])
        b = tl.load(b_ptr + offs_k[:, None] * N + offs_n[None, :])

        # matrix multiply
        acc += tl.dot(a, b)

    # store result
    tl.store(c_ptr + offs_m[:, None] * N + offs_n[None, :], acc.to(tl.float16))

def simple_tiled_matmul(a, BLOCK=32):
    N = a.shape[0]

    c = torch.empty((N, N), device=a.device, dtype=torch.float16)

    grid = ((N // BLOCK) * (N // BLOCK),)

    simple_tiled_matmul_kernel[grid](
        a, a, c,
        N,
        BLOCK=BLOCK,
        num_warps = 4,
    )

    return c

1. What block size leads to failure (block launch impossible as it exceeds shared memory capacity)
2. What block size (in powers of 2) and size of matrix leads to highest throughput?
3. Assuming FP32, what is the SRAM Usage per SM for the above block size
4. Is this kernel compute bound or memory bound
5. num warps = 4 leads to best occupancy. Why?

In [58]:
N = 2048
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5

# Run kernel
t = bench(lambda: simple_tiled_matmul(A, 64), 50, 200)
#t = bench(lambda: torch.matmul(A,A))

print(f"Kernel execution time: {t} ms")
print(f"Gflops: {gflops(N, t)} ")

Kernel execution time: 6.040899753570557 ms
Gflops: 2843.925554938336 


In [38]:
import torch.profiler as profiler

with profiler.profile(
    activities=[profiler.ProfilerActivity.CPU, profiler.ProfilerActivity.CUDA],
    record_shapes=True
) as prof:
    bench(lambda: simple_tiled_matmul(A, 64), 10, 50)

print(prof.key_averages().table(sort_by="cuda_time_total"))

------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                          Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
    simple_tiled_matmul_kernel         0.00%       0.000us         0.00%       0.000us       0.000us     415.937ms       100.00%     415.937ms       7.427ms            56  
                   aten::empty         0.08%     370.780us         0.08%     370.780us       6.180us       0.000us         0.00%       0.000us       0.000us            60  
       Activity Buffer Request         0.20%     889.010us         0.20%     889.010us     889.010us       0.000us         0.00%      